# Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.1-8b-instant")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots are known for their ability to mimic human speech and other sounds they hear in their environment. This behavior is called vocal learning or vocal mimicry. Parrots have a unique anatomy and brain structure that allows them to produce a wide range of sounds, including speech-like vocalizations.\n\nThere are several reasons why parrots talk:\n\n1. **Social bonding**: Parrots are highly social animals that live in flocks in the wild. They use vocalizations to communicate with each other, establish social bonds, and maintain relationships within their group. By mimicking human speech, they may be attempting to form a bond with their human caregivers.\n2. **Attention seeking**: Parrots may talk to get attention from their owners or to initiate interaction. They may learn to associate talking with getting treats, praise, or affection.\n3. **Imitation**: Parrots are known for their ability to imitate sounds they hear in their environment. They may learn to mimic hum

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'tebwr4kae', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 220, 'total_tokens': 234, 'completion_time': 0.029189859, 'completion_tokens_details': None, 'prompt_time': 0.019158692, 'prompt_tokens_details': None, 'queue_time': 0.005923855, 'total_time': 0.048348551}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fbe80-d6fc-7912-9a2d-92379636f4f9-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'tebwr4kae', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 14, 'total_tokens': 234}
Tool: get_weather
Args: {'location': 'Boston'}


# Tool Execution Loop

In [9]:
messages = [
    {"role": "user", "content": "What's the weather like in Boston?"}
]

response = model_with_tools.invoke(messages)
messages.append(response)

for tool_call in response.tool_calls:
    tool_result = get_weather.invoke(tool_call)

    print("Tool result:", tool_result.content)

    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)

print("Model response:", final_response.content)

Tool result: It's sunny in Boston
Model response: I need more information about Boston weather to give you more accurate information.


In [8]:
messages

[{'role': 'user', 'content': "What's the weather like in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9a150t7kz', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 220, 'total_tokens': 234, 'completion_time': 0.033788578, 'completion_tokens_details': None, 'prompt_time': 0.022218535, 'prompt_tokens_details': None, 'queue_time': 0.006832185, 'total_time': 0.056007113}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fbefb-fa70-7673-bcd3-9a5bc79cd69f-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '9a150t7kz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 14, 'total_tokens': 234}),
 ToolMessage(content="It